In [ ]:
import torch
import torch.nn as nn
from torchsummary import summary

class TinyYOLO_FC(nn.Module):  # model for images of size 224 × 224
    def __init__(self, S=7, B=2, C=13):
        super().__init__()
        self.S = S # number of grid cells
        self.B = B # number of bounding boxes per grid cell -> basically number of elements to try to detect in each grid cell
        self.C = C # number of classes to detect
        self.output_dim = S * S * (B * 5 + C) 

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),                         # 256 × 7 × 7 = 12544
            nn.Linear(12544, 768), nn.ReLU(),     
            nn.Linear(768, self.output_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        x = x.view(-1, self.S, self.S, self.B * 5 + self.C)
        return x


model = TinyYOLO_FC()
summary(model, (3, 224, 224), device="cpu")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 224, 224]             448
              ReLU-2         [-1, 16, 224, 224]               0
         MaxPool2d-3         [-1, 16, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]           4,640
              ReLU-5         [-1, 32, 112, 112]               0
         MaxPool2d-6           [-1, 32, 56, 56]               0
            Conv2d-7           [-1, 64, 56, 56]          18,496
              ReLU-8           [-1, 64, 56, 56]               0
         MaxPool2d-9           [-1, 64, 28, 28]               0
           Conv2d-10          [-1, 128, 28, 28]          73,856
             ReLU-11          [-1, 128, 28, 28]               0
        MaxPool2d-12          [-1, 128, 14, 14]               0
           Conv2d-13          [-1, 256, 14, 14]         295,168
             ReLU-14          [-1, 256,

Dataset Loader 

In [7]:
import pandas as pd
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T

class YoloCSVDataset(Dataset):
    def __init__(self, csv_file, image_dir, S=7, B=2, C=13, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.S = S
        self.B = B
        self.C = C
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor()
        ])
        self.image_files = self.annotations['filename'].unique()

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        img_name = self.image_files[index]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        boxes = self.annotations[self.annotations['filename'] == img_name]
        target = torch.zeros((self.S, self.S, self.B * 5 + self.C))

        for _, row in boxes.iterrows():
            cls, x, y, w, h = int(row['class']), row['x_center'], row['y_center'], row['width'], row['height']
            i, j = int(self.S * y), int(self.S * x)
            x_cell, y_cell = self.S * x - j, self.S * y - i
            for b in range(self.B):
                target[i, j, b*5:b*5+5] = torch.tensor([x_cell, y_cell, w, h, 1])
            target[i, j, self.B * 5 + cls] = 1
        return image, target
